In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [2]:
!pip install -q transformers datasets accelerate scikit-learn

In [3]:
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
from datasets import load_dataset

ds = load_dataset("yaful/MAGE")

print(ds)
print(ds["train"].column_names)

README.md:   0%|          | 0.00/1.93k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B /  404MB            

train.csv: downloading bytes:           |  0.00B            

valid.csv: reconstructing file:   0%|          |  0.00B / 72.3MB            

valid.csv: downloading bytes:           |  0.00B            

test.csv: reconstructing file:   0%|          |  0.00B / 71.7MB            

... (3 outputs left out)

Generating train split:   0%|          | 0/319071 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/56792 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/60743 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'src'],
        num_rows: 319071
    })
    validation: Dataset({
        features: ['text', 'label', 'src'],
        num_rows: 56792
    })
    test: Dataset({
        features: ['text', 'label', 'src'],
        num_rows: 60743
    })
})
['text', 'label', 'src']


In [5]:
from datasets import Dataset
import pandas as pd

def clean_split(split):
    df = split.to_pandas()

    # Remove missing and blank text
    df = df.dropna(subset=["text"])
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["text"] != ""]

    # Deduplicate within this split
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
    return df

train_df = clean_split(ds["train"])
val_df = clean_split(ds["validation"])
test_df = clean_split(ds["test"])

# Remove validation examples that also occur in test
test_texts = set(test_df["text"])
val_df = val_df[~val_df["text"].isin(test_texts)].reset_index(drop=True)

# Balance training: retain all human examples, sample equal AI examples
human_df = train_df[train_df["label"] == 1]
ai_df = train_df[train_df["label"] == 0]

ai_sample = ai_df.sample(
    n=len(human_df),
    random_state=42
)

balanced_train_df = (
    pd.concat([human_df, ai_sample], ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("Balanced train:", balanced_train_df.shape)
print(balanced_train_df["label"].value_counts())
print("Validation:", val_df.shape, val_df["label"].value_counts().to_dict())
print("Test:", test_df.shape, test_df["label"].value_counts().to_dict())

Balanced train: (186636, 3)
label
0    93318
1    93318
Name: count, dtype: int64
Validation: (56740, 3) {1: 28749, 0: 27991}
Test: (59981, 3) {0: 30478, 1: 29503}


In [6]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(
    balanced_train_df[["text", "label"]], preserve_index=False
)
val_ds = Dataset.from_pandas(
    val_df[["text", "label"]], preserve_index=False
)
test_ds = Dataset.from_pandas(
    test_df[["text", "label"]], preserve_index=False
)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_ds = train_ds.map(
    tokenize_batch, batched=True, remove_columns=["text"]
)
val_ds = val_ds.map(
    tokenize_batch, batched=True, remove_columns=["text"]
)
test_ds = test_ds.map(
    tokenize_batch, batched=True, remove_columns=["text"]
)

print(train_ds)
print(val_ds)
print(test_ds)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/186636 [00:00<?, ? examples/s]

Map:   0%|          | 0/56740 [00:00<?, ? examples/s]

Map:   0%|          | 0/59981 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 186636
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 56740
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 59981
})


In [7]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "roc_auc": roc_auc_score(labels, probs[:, 1])
    }

total_steps = (
    len(train_ds)
    // (16 * 2)  # batch size × gradient accumulation
) * 3

training_args = TrainingArguments(
    output_dir="./distilbert_mage",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=int(0.1 * total_steps),
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_roc_auc",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=500,
    report_to="none",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Roc Auc
1,0.343612,0.308123,0.875291,0.883275,0.876266,0.874821,0.967269
2,0.224940,0.384933,0.882305,0.894387,0.883488,0.881609,0.976617
3,0.125978,0.530013,0.893144,0.900698,0.894078,0.892784,0.974932


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=17499, training_loss=0.30174701432785556, metrics={'train_runtime': 3706.2516, 'train_samples_per_second': 151.071, 'train_steps_per_second': 4.721, 'total_flos': 3.707060825129184e+16, 'train_loss': 0.30174701432785556, 'epoch': 3.0})

In [9]:
from sklearn.metrics import classification_report, confusion_matrix

test_output = trainer.predict(test_ds)

test_logits = test_output.predictions
test_labels = test_output.label_ids

test_probs = torch.softmax(
    torch.tensor(test_logits), dim=1
).numpy()

test_preds = np.argmax(test_probs, axis=1)

print("Test Accuracy:", accuracy_score(test_labels, test_preds))
print("Test ROC-AUC:", roc_auc_score(test_labels, test_probs[:, 1]))

print("\nClassification Report:")
print(classification_report(
    test_labels,
    test_preds,
    target_names=["AI-generated", "Human-written"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(test_labels, test_preds))


Test Accuracy: 0.8798619562861573
Test ROC-AUC: 0.9724257777729476

Classification Report:
               precision    recall  f1-score   support

 AI-generated       0.83      0.97      0.89     30478
Human-written       0.96      0.79      0.87     29503

     accuracy                           0.88     59981
    macro avg       0.89      0.88      0.88     59981
 weighted avg       0.89      0.88      0.88     59981


Confusion Matrix:
[[29486   992]
 [ 6214 23289]]


In [10]:
import os

SAVE_DIR = "/content/distilbert_mage_final"
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
trainer.save_state()

print("Model and tokenizer saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved!


In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [13]:
import json
import pandas as pd
import numpy as np

results = {
    "accuracy": float(accuracy_score(test_labels, test_preds)),
    "macro_f1": float(f1_score(test_labels, test_preds, average="macro")),
    "roc_auc": float(roc_auc_score(test_labels, test_probs[:, 1]))
}

with open(f"{SAVE_DIR}/test_metrics.json", "w") as f:
    json.dump(results, f, indent=4)

predictions_df = pd.DataFrame({
    "true_label": test_labels,
    "predicted_label": test_preds,
    "ai_probability": test_probs[:, 0],
    "human_probability": test_probs[:, 1]
})

predictions_df.to_csv(
    f"{SAVE_DIR}/test_predictions.csv",
    index=False
)

print(results)

{'accuracy': 0.8798619562861573, 'macro_f1': 0.8785657419756966, 'roc_auc': 0.9724257777729476}


In [14]:
import shutil

ZIP_PATH = shutil.make_archive(
    "/content/distilbert_mage_final",
    "zip",
    root_dir="/content",
    base_dir="distilbert_mage_final"
)

print("Created:", ZIP_PATH)

Created: /content/distilbert_mage_final.zip


In [15]:
from google.colab import files

files.download(ZIP_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>